# Customer behaviour Analysis (Cleaning & upload in Database)


In [1]:
import pandas as pd

df= pd.read_csv("C:\\Users\\ssahoo7\\Desktop\\p2\\customer_behavior.csv")

In [2]:
df.sample(5)

,Order ID,Customer ID,Signup Date,Purchase Date,Purchase Time,Age,Name,Item Purchased,Category,Quantity,...,Discount Applied,Promo Used,Preferred Payment Method,Frequency,Channel,Campaign,Loyality member,Member ship join date,Loyality Points,Redem
48735,48736,3554,2/17/2022,6/28/2023,19:15,48,Mr. Pooja Verma,Lenovo Tab M10,Lenovo Tablet,1,...,No,Yes,Net Banking,Quarterly,Mobile App,New Launch,No,NaN,0,0.0
16864,16865,515,1/19/2022,2/27/2024,18:45,30,Mr. Manish Mishra,HP Keyboard,HP Accessories,1,...,No,Yes,Net Banking,Annually,Web,Exchange Offer,No,NaN,0,0.0
73383,73384,2925,7/17/2022,10/15/2022,21:15,26,Miss Anjali Jain,Lenovo L24i,Lenovo Monitor,1,...,No,Yes,Credit Card,Quarterly,Store,Student Offer,No,NaN,0,0.0
77445,77446,2002,1/8/2022,12/7/2022,11:15,20,Miss Meena Jain,Dell Inspiron AIO 24,Dell All in One,1,...,Yes,No,Debit Card,Monthly,Mobile App,Festive Sale,No,NaN,0,0.0
49775,49776,1808,2/26/2022,8/27/2023,16:00,26,Mr. Priya Singh,HP Pro Tablet 608,HP Tablet,1,...,No,No,Net Banking,Weekly,Mobile App,No Campaign,No,NaN,0,0.0


In [3]:
df.dtypes

Order ID                      int64
Customer ID                   int64
Signup Date                  object
Purchase Date                object
Purchase Time                object
Age                           int64
Name                         object
Item Purchased               object
Category                     object
Quantity                      int64
Revenue                      object
Cost                         object
Location                     object
Configuration                object
Color Variant                object
Review Rating               float64
Subscription Status          object
Payment Method               object
Card Bank                    object
EMI                          object
EMI Tenure                   object
Shipping Type                object
Discount Applied             object
Promo Used                   object
Preferred Payment Method     object
Frequency                    object
Channel                      object
Campaign                    

In [4]:
df.columns

Index(['Order ID', 'Customer ID', 'Signup Date', 'Purchase Date',
       'Purchase Time', 'Age', 'Name', 'Item Purchased', 'Category',
       'Quantity', 'Revenue', 'Cost', 'Location', 'Configuration',
       'Color Variant', 'Review Rating', 'Subscription Status',
       'Payment Method', 'Card Bank', 'EMI', 'EMI Tenure', 'Shipping Type',
       'Discount Applied', 'Promo Used', 'Preferred Payment Method',
       'Frequency', 'Channel', 'Campaign', 'Loyality member',
       'Member ship join date', 'Loyality Points', 'Redem'],
      dtype='object')

In [5]:
# handle data types
df['Signup Date']= pd.to_datetime(df['Signup Date'])
df['Purchase Date']= pd.to_datetime(df['Purchase Date'])
df['Member ship join date']= pd.to_datetime(df['Member ship join date'])

df['Purchase Time'] = pd.to_datetime(df['Purchase Time'], errors='coerce').dt.time

df['Name']= df['Name'].astype(str)

df['Revenue']= df['Revenue'].str.split('.').str.get(0).str.replace('₹','').str.replace(',','').astype(int)
df['Cost'] = df['Cost'].str.split('.').str.get(0).str.replace('₹','').str.replace(',','').astype(int)

df['EMI Tenure']= df['EMI Tenure'].str.replace("M","").fillna(0).astype(int)

df["Redem"].isna().sum()
df['Redem']= df['Redem'].fillna(0).astype(int)

C:\Users\ssahoo7\AppData\Local\Temp\ipykernel_29420\3314399396.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Purchase Time'] = pd.to_datetime(df['Purchase Time'], errors='coerce').dt.time


In [6]:
# Creating Gender column
def gender(name):
    name = str(name)  # prevent error if NaN
    
    if "Mrs." in name:
        return "Female"
    elif "Miss" in name:
        return "Female"
    elif "Mr." in name:
        return "Male"
    else:
        return "Others"

Gender = df['Name'].apply(gender)
df.insert(6, "Gender", Gender)

df.drop(columns= "Name", inplace = True)

In [7]:
# Creating Brand column
df['Brand'] = df['Item Purchased'].str.split(" ").str.get(0)

df['Brand'].replace({
    'iMac': "Apple",
    "ThinkCentre": "Lenovo",
    "IdeaCentre": "Lenovo",
    "MacBook": "Apple",
    "Mac": "Apple",
    "Magic": "Apple",
    "ThinkVision": "Lenovo",
    "ThinkPad": "Lenovo",
    "IdeaPad": "Lenovo",
    "Legion": "Lenovo",
    "AirPods": "Apple",
    "iPad":"Apple"
}, inplace=True)


brand_col = df.pop('Brand')        
df.insert(6, 'Brand', brand_col)

C:\Users\ssahoo7\AppData\Local\Temp\ipykernel_29420\1061865611.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Brand'].replace({


In [8]:
# Splitting configuration column in- Processor, Ram, Memory, OS cokumn
processor= df['Configuration'].str.split("/").str.get(0)
ram= df['Configuration'].str.split("/").str.get(1).str.replace("GB","").astype(int)
memory= df['Configuration'].str.split("/").str.get(2).str.replace("GB|TB","")
os= df['Configuration'].str.split("/").str.get(3)
graphics= df['Configuration'].str.split("/").str.get(4)

df.insert(13, "Processor",processor)
df.insert(14, "Ram", ram)
df.insert(15, "Memory", memory)
df.insert(16, "OS", os)
df.insert(17, "Graphics", graphics)

In [9]:
# replacing the blank
df['Graphics']= df['Graphics'].fillna("Integrated")

In [10]:
# Add formfactor column
product= df['Category'].str.split(" ").str.get(1).str.strip()
df.insert(7,"Formfactor",product)

In [11]:
df['Formfactor']= df['Formfactor'].str.replace('All','All-in-One')

In [12]:
# Handling Memory column
def convert_memory(x):
    if 'TB' in str(x):
        return float(x.replace('TB','')) * 1000
    elif 'GB' in str(x):
        return float(x.replace('GB',''))
    else:
        return pd.to_numeric(x, errors='coerce')

df['Memory'] = df['Memory'].apply(convert_memory).astype(int)

In [13]:
# Creating Profit column
Profit= df['Revenue'] - df['Cost']

df.insert(12, "Profit", Profit)

In [14]:
# Creating Gross Profit(%) column
gross_profit= df["Profit"] / df["Revenue"] *100

df.insert(13, "Gross Profit", gross_profit)

Data Validation

In [15]:
# data validation
df[(df['Loyality member']==0) & (df['Loyality Points']>0)]

# Finding duplicate orders
df.duplicated(subset=['Order ID']).sum()

# Rounding up the Gross profit column
df['Gross Profit'].fillna(0).round(0)

# age outlier check
df[df['Age'] < 15]
df[df['Age'] > 80]

# Data type change
df['Ram']= df[['Ram']].astype(int)
df['Memory']= df[['Memory']].astype(int)

# fill up the graphics column
df["Graphics"].fillna("Intigrated", inplace= True)

C:\Users\ssahoo7\AppData\Local\Temp\ipykernel_29420\3337977802.py:19: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Graphics"].fillna("Intigrated", inplace= True)


In [16]:
# Fix rows where Signup > Member ship Join Date
mask = (
    df['Signup Date'].notna() &
    df['Member ship join date'].notna() &
    (df['Signup Date'] > df['Member ship join date'])
)

df.loc[mask, 'Member ship join date'] = df.loc[mask, 'Signup Date']

# Ensure ONE join date per customer (earliest signup) ONLY for customers who are loyalty members
# get min signup per customer
min_signup = df.groupby('Customer ID')['Signup Date'].transform('min')

# apply only where loyalty member = Yes
df.loc[df['Loyality member'] == 'Yes', 'Member ship join date'] = \
    min_signup[df['Loyality member'] == 'Yes']

validation = (
    df[df['Member ship join date'].notna()]
    .assign(min_signup = df.groupby('Customer ID')['Signup Date'].transform('min'))
    .query("min_signup != `Member ship join date`")
)

In [17]:
# Check signup date > purchase date
df[df['Signup Date'] > df['Purchase Date']]

,Order ID,Customer ID,Signup Date,Purchase Date,Purchase Time,Age,Brand,Formfactor,Gender,Item Purchased,...,Discount Applied,Promo Used,Preferred Payment Method,Frequency,Channel,Campaign,Loyality member,Member ship join date,Loyality Points,Redem


In [18]:
# One Membership Date per Customer
(df.groupby('Customer ID')['Member ship join date'].nunique() > 1).sum()

np.int64(0)

In [19]:
# Loyalty Yes but Join Date Blank
df[(df['Loyality member']=='Yes') & (df['Member ship join date'].isna())].shape[0]

0

In [20]:
# Profit = Revenue - Cost
((df['Revenue'] - df['Cost']) != df['Profit']).sum()

np.int64(0)

In [21]:
# Redeem > Loyalty Points (Logical Issue?)
(df['Redem'] > df['Loyality Points']).sum()

np.int64(0)

In [22]:
# Purchase Date < Membership Join Date
(df['Purchase Date'] < df['Member ship join date']).sum()

np.int64(0)

Others

In [23]:
# Lowercase to the column headers   
df.columns= df.columns.str.lower()

In [24]:
df.drop(columns='configuration', inplace=True)

In [25]:
df.to_csv('final.csv', index=False)

# Load Data to MySQL

In [40]:
from sqlalchemy import create_engine

server = r"SSAHOO7-W0CVR6W\SQLEXPRESS"   # your server
database = "customer_analytics"

connection_str = (
    f"mssql+pyodbc://@{server}/{database}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
)

engine = create_engine(connection_str)

# Test connection first
engine.connect()

print("Connected successfully to SQL Server")

df.to_sql(
    name="customer_behavior",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=5000
)

print("Data uploaded to SQL Server")

C:\Users\ssahoo7\AppData\Local\Temp\ipykernel_33520\2600732077.py:15: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  engine.connect()


Connected successfully to SQL Server
Data uploaded to SQL Server
